Final implementation of P300NN with the proper optimizer and hyperparameters to compare against other P300 prediction types including a static Baseline and a Linear Discriminant Analysis Model. 

In [5]:
import mne
import numpy as np
from numpy import matlib as mb
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import warnings
from sklearn import metrics
from collections import defaultdict
import regex as re
from sklearn.metrics import accuracy_score
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)
from sklearn.metrics import roc_auc_score
from models.P300NN import P300NNClassifier
from models.P300NN_noClassBalance import P300NNClassifierNCB
from src.Dataset import Dataset
from src.Speller import Speller
warnings.filterwarnings("ignore", category=RuntimeWarning)
mne.set_log_level("WARNING")
import tensorflow as tf
print("GPUs available:", tf.config.list_physical_devices('GPU'))
print("CPUs available:", tf.config.list_physical_devices('CPU'))
print("TF use:", tf.test.gpu_device_name() if tf.test.gpu_device_name() else "CPU")

GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
CPUs available: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
TF use: /device:GPU:0


I0000 00:00:1777097860.625038 1716639 gpu_device.cc:2043] Created device /device:GPU:0 with 596 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:5e:00.0, compute capability: 7.5
I0000 00:00:1777097860.634057 1716639 gpu_device.cc:2043] Created device /device:GPU:0 with 596 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 2080 Ti, pci bus id: 0000:5e:00.0, compute capability: 7.5


Divide and Separate Training and Testing Paths

In [6]:
def group_paths_by_participant(data_paths):
    participant_files = defaultdict(lambda: {'train': [], 'test': []})
    for path in data_paths:
        parts = path.split(os.sep)
        participant_id = parts[6]

        if 'Train' in path or 'train' in path:
            participant_files[participant_id]['train'].append(path)
        elif 'Test' in path or 'test' in path:
            participant_files[participant_id]['test'].append(path)
        else:
            print(f"Unclassified path: {path}")

    return participant_files


In [7]:
important_channels = ['EEG_Fz', 'EEG_Cz', 'EEG_Pz', 'EEG_P3',
                      'EEG_PO7', 'EEG_PO8', 'EEG_P4', 'EEG_Oz']

#first using default values
ds = Dataset(
    glob_path=os.path.join(project_root, "data", "*", "*", "*", "CB", "*"),
    tmin=0,
    tmax=0.8, 
)

participant_files = group_paths_by_participant(ds.data_paths)
participants = list(participant_files.keys())

In [9]:
#With Preprocessing

studyL_results = []

# Loop through all participants
for participant_id in participant_files.keys():

    print(f"\nProcessing participant {participant_id}")

    train_files = participant_files[participant_id]['train']
    test_files  = participant_files[participant_id]['test']

    X_train, y_train = [], []
    X_test, y_test   = [], []

    # Load train data
    for path in train_files:
        ds = Dataset(
            path,
            important_channels=important_channels,
            sample_rate=256,
            tmin=0,
            tmax=0.8,
            use_car=True, 
            notch_filter=True
        )
        X, y = ds[0]
        X_train.append(X)
        y_train.append(y)

    # Load test data
    for path in test_files:
        ds = Dataset(
            path,
            important_channels=important_channels,
            sample_rate=256,
            tmin=0,
            tmax=0.8,
            use_car=True, 
            notch_filter=True
        )
        X, y = ds[0]
        X_test.append(X)
        y_test.append(y)

    X_train = np.vstack(X_train)
    y_train = np.concatenate(y_train)

    X_test = np.vstack(X_test)
    y_test = np.concatenate(y_test)

    X_train_nn = X_train[..., np.newaxis]
    X_test_nn = X_test[..., np.newaxis]

    # Train on train partition
    p300net = P300NNClassifier(dropout_rate=0.3, learning_rate=0.001, optim_type = 'adam', F1 = 16)
    p300net.train(X_train_nn, y_train)

    # Test on test partition
    scores = p300net.test(X_test_nn, y_test)
    y_pred = np.argmax(p300net.model.predict(X_test_nn), axis=1)
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, scores)

    print(f"Participant {participant_id} P300net Accuracy: {acc:.3f}, AUC: {auc:.3f}")

    # Decode and predict characters
    character_accuracies = []
    for path in test_files:
        ds = Dataset(
            path,
            important_channels=important_channels,
            sample_rate=256,
            tmin=0,
            tmax=0.8, 
            notch_filter=True, 
            use_car=True
        )
        X_test_specific, y_test_specific = ds[0]
        X_test_specific_nn = X_test_specific[..., np.newaxis]

        probs = p300net.model.predict(X_test_specific_nn)
        scores_specific = probs[:, 1] - probs[:, 0]

        # Speller implementation
        raw = ds.raw
        epochs = ds.epochs
        char_ch_names = [ch for ch in raw.ch_names if re.match(r'^[A-Za-z0-9]+_\d+_\d+$', ch)]
        char_ch_indices = [raw.ch_names.index(ch) for ch in char_ch_names]

        curr_target_idx = raw.ch_names.index('CurrentTarget')
        phase_idx = raw.ch_names.index('PhaseInSequence')
        data = raw.get_data()
        stim_indices = np.where(data[phase_idx] == 2)[0]
        changes = np.diff(data[curr_target_idx], prepend=data[curr_target_idx][0]-1)
        target_onsets = stim_indices[np.isin(stim_indices, np.where(changes != 0)[0])]
        target_codes = data[curr_target_idx][target_onsets].astype(int)
        current_target_events = np.array([[onset, 0, code] for onset, code in zip(target_onsets, target_codes)])

        pcr = {
            'epochs': epochs,
            'raw_data': raw,
            'character_channels': char_ch_indices,
            'current_target_events': current_target_events
        }

        speller_grid = [
            "A","B","C","D","E","F","G","H","I","J","K","L","M",
            "N","O","P","Q","R","S","T","U","V","W","X","Y","Z",
            "_","1","2","3","4","5","6","7","8","9"
        ]
        speller = Speller(speller_grid)

        predictions = speller.run(pcr, clf=None, X=scores_specific, y=1)
        metrics = speller.get_metrics()
        character_accuracies.append(metrics['accuracy'])

    avg_char_acc = np.mean(character_accuracies)
    print(f"Average character accuracy: {avg_char_acc:.3f}")

    # Store results
    studyL_results.append({
        "participant": participant_id,
        "p300net_accuracy": acc,
        "p300net_auc": auc,
        "characcuracy": avg_char_acc,
    })

# Summary DataFrame
results_df = pd.DataFrame(studyL_results)
print("\nStudy L Summary")
print(results_df)
print("Average AUC:", results_df["p300net_auc"].mean())
print("Average Model Accuracy:", results_df["p300net_accuracy"].mean())
print("Average Character Accuracy:", results_df["characcuracy"].mean())


Processing participant L_01
Epoch 1/3000


I0000 00:00:1777099824.765906 1721617 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_431695__.48
I0000 00:00:1777099827.845913 1721617 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_431695__.48


119/119 - 8s - 69ms/step - accuracy: 0.5180 - loss: 0.6932
Epoch 2/3000
119/119 - 4s - 34ms/step - accuracy: 0.6471 - loss: 0.6827
Epoch 3/3000
119/119 - 1s - 5ms/step - accuracy: 0.6354 - loss: 0.6681
Epoch 4/3000
119/119 - 1s - 5ms/step - accuracy: 0.6151 - loss: 0.6620
Epoch 5/3000
119/119 - 1s - 5ms/step - accuracy: 0.6413 - loss: 0.6498
Epoch 6/3000
119/119 - 1s - 5ms/step - accuracy: 0.6450 - loss: 0.6483
Epoch 7/3000
119/119 - 1s - 5ms/step - accuracy: 0.6595 - loss: 0.6396
Epoch 8/3000
119/119 - 1s - 5ms/step - accuracy: 0.6474 - loss: 0.6409
Epoch 9/3000
119/119 - 1s - 5ms/step - accuracy: 0.6669 - loss: 0.6251
Epoch 10/3000
119/119 - 1s - 5ms/step - accuracy: 0.6693 - loss: 0.6266
Epoch 11/3000
119/119 - 1s - 5ms/step - accuracy: 0.6664 - loss: 0.6277
Epoch 12/3000
119/119 - 1s - 5ms/step - accuracy: 0.6966 - loss: 0.6170
Epoch 13/3000
119/119 - 1s - 5ms/step - accuracy: 0.6807 - loss: 0.6194
Epoch 14/3000
119/119 - 1s - 5ms/step - accuracy: 0.6780 - loss: 0.6038
Epoch 15/300

I0000 00:00:1777099855.454927 1721617 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_449731__.48
I0000 00:00:1777099858.488522 1721617 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_449731__.48


119/119 - 8s - 69ms/step - accuracy: 0.5153 - loss: 0.6940
Epoch 2/3000
119/119 - 4s - 34ms/step - accuracy: 0.6164 - loss: 0.6898
Epoch 3/3000
119/119 - 1s - 5ms/step - accuracy: 0.5317 - loss: 0.6846
Epoch 4/3000
119/119 - 1s - 5ms/step - accuracy: 0.5529 - loss: 0.6819
Epoch 5/3000
119/119 - 1s - 5ms/step - accuracy: 0.5669 - loss: 0.6725
Epoch 6/3000
119/119 - 1s - 5ms/step - accuracy: 0.6214 - loss: 0.6666
Epoch 7/3000
119/119 - 1s - 5ms/step - accuracy: 0.6143 - loss: 0.6561
Epoch 8/3000
119/119 - 1s - 5ms/step - accuracy: 0.6418 - loss: 0.6430
Epoch 9/3000
119/119 - 1s - 6ms/step - accuracy: 0.6437 - loss: 0.6254
Epoch 10/3000
119/119 - 1s - 5ms/step - accuracy: 0.6643 - loss: 0.6212
Epoch 11/3000
119/119 - 1s - 5ms/step - accuracy: 0.6444 - loss: 0.6157
Epoch 12/3000
119/119 - 1s - 5ms/step - accuracy: 0.6706 - loss: 0.6052
Epoch 13/3000
119/119 - 1s - 5ms/step - accuracy: 0.6709 - loss: 0.6138
Epoch 14/3000
119/119 - 1s - 5ms/step - accuracy: 0.6537 - loss: 0.6179
Epoch 15/300

...

119/119 - 8s - 70ms/step - accuracy: 0.5513 - loss: 0.6926
Epoch 2/3000
119/119 - 4s - 33ms/step - accuracy: 0.6349 - loss: 0.6428
Epoch 3/3000
119/119 - 1s - 5ms/step - accuracy: 0.6519 - loss: 0.6223
Epoch 4/3000
119/119 - 1s - 5ms/step - accuracy: 0.6757 - loss: 0.6018
Epoch 5/3000
119/119 - 1s - 5ms/step - accuracy: 0.7056 - loss: 0.5809
Epoch 6/3000
119/119 - 1s - 5ms/step - accuracy: 0.7140 - loss: 0.5724
Epoch 7/3000
119/119 - 1s - 5ms/step - accuracy: 0.7135 - loss: 0.5783
Epoch 8/3000
119/119 - 1s - 5ms/step - accuracy: 0.7270 - loss: 0.5611
Epoch 9/3000
119/119 - 1s - 5ms/step - accuracy: 0.7169 - loss: 0.5660
Epoch 10/3000
119/119 - 1s - 5ms/step - accuracy: 0.7460 - loss: 0.5515
Epoch 11/3000
119/119 - 1s - 5ms/step - accuracy: 0.7399 - loss: 0.5507
Epoch 12/3000
119/119 - 1s - 5ms/step - accuracy: 0.7452 - loss: 0.5531
Epoch 13/3000
119/119 - 1s - 5ms/step - accuracy: 0.7360 - loss: 0.5351
Epoch 14/3000
119/119 - 1s - 5ms/step - accuracy: 0.7381 - loss: 0.5354
Epoch 15/300

I0000 00:00:1777100167.781109 1721617 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_651939__.48
I0000 00:00:1777100170.829882 1721617 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_651939__.48


119/119 - 8s - 69ms/step - accuracy: 0.7106 - loss: 0.6938
Epoch 2/3000
119/119 - 4s - 34ms/step - accuracy: 0.4196 - loss: 0.6932
Epoch 3/3000
119/119 - 1s - 5ms/step - accuracy: 0.5624 - loss: 0.6880
Epoch 4/3000
119/119 - 1s - 5ms/step - accuracy: 0.5569 - loss: 0.6805
Epoch 5/3000
119/119 - 1s - 5ms/step - accuracy: 0.6019 - loss: 0.6734
Epoch 6/3000
119/119 - 1s - 5ms/step - accuracy: 0.6307 - loss: 0.6678
Epoch 7/3000
119/119 - 1s - 5ms/step - accuracy: 0.6045 - loss: 0.6658
Epoch 8/3000
119/119 - 1s - 5ms/step - accuracy: 0.6376 - loss: 0.6474
Epoch 9/3000
119/119 - 1s - 5ms/step - accuracy: 0.6357 - loss: 0.6515
Epoch 10/3000
119/119 - 1s - 5ms/step - accuracy: 0.6450 - loss: 0.6506
Epoch 11/3000
119/119 - 1s - 5ms/step - accuracy: 0.6349 - loss: 0.6361
Epoch 12/3000
119/119 - 1s - 5ms/step - accuracy: 0.6503 - loss: 0.6454
Epoch 13/3000
119/119 - 1s - 5ms/step - accuracy: 0.6233 - loss: 0.6326
Epoch 14/3000
119/119 - 1s - 5ms/step - accuracy: 0.6270 - loss: 0.6446
Epoch 15/300

I0000 00:00:1777100209.456231 1721617 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_680131__.48
I0000 00:00:1777100212.517767 1721617 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_680131__.48


119/119 - 8s - 69ms/step - accuracy: 0.5042 - loss: 0.6859
Epoch 2/3000
119/119 - 4s - 34ms/step - accuracy: 0.6841 - loss: 0.6350
Epoch 3/3000
119/119 - 1s - 5ms/step - accuracy: 0.6635 - loss: 0.6070
Epoch 4/3000
119/119 - 1s - 5ms/step - accuracy: 0.6947 - loss: 0.5991
Epoch 5/3000
119/119 - 1s - 5ms/step - accuracy: 0.6852 - loss: 0.5833
Epoch 6/3000
119/119 - 1s - 5ms/step - accuracy: 0.7101 - loss: 0.5733
Epoch 7/3000
119/119 - 1s - 5ms/step - accuracy: 0.6981 - loss: 0.5688
Epoch 8/3000
119/119 - 1s - 5ms/step - accuracy: 0.7085 - loss: 0.5629
Epoch 9/3000
119/119 - 1s - 5ms/step - accuracy: 0.7135 - loss: 0.5628
Epoch 10/3000
119/119 - 1s - 5ms/step - accuracy: 0.7045 - loss: 0.5663
Epoch 11/3000
119/119 - 1s - 5ms/step - accuracy: 0.7212 - loss: 0.5461
Epoch 12/3000
119/119 - 1s - 5ms/step - accuracy: 0.7241 - loss: 0.5409
Epoch 13/3000
119/119 - 1s - 5ms/step - accuracy: 0.7230 - loss: 0.5404
Epoch 14/3000
119/119 - 1s - 5ms/step - accuracy: 0.7368 - loss: 0.5325
Epoch 15/300